In [0]:
host = dbutils.secrets.get("hpn-db", "kv-postgres-host")
pwd  = dbutils.secrets.get("hpn-db", "kv-postgres-pwd")
user  = dbutils.secrets.get("hpn-db", "kv-postgres-user")
db  = dbutils.secrets.get("hpn-db", "kv-postgres-db")

# # não-sensíveis → config (pega no dashboard do Neon → Connection Details)
# user     = "SEU_USUARIO_NEON"     # <- ajuste
# database = "SEU_DATABASE_NEON"    # <- ajuste (ex. neondb)

url = f"jdbc:postgresql://{host}:5432/{db}?sslmode=require"

df = (spark.read.format("jdbc")
      .option("url", url)
      .option("user", user)
      .option("password", pwd)
      .option("driver", "org.postgresql.Driver")
      .option("query", """
          SELECT table_schema, table_name
          FROM information_schema.tables
          WHERE table_schema NOT IN ('pg_catalog','information_schema')
          ORDER BY 1,2
      """)
      .load())

display(df)


In [0]:
%sql

-- schema pra metadados

CREATE SCHEMA IF NOT EXISTS hpn.metadata
  COMMENT 'Objetos de controle/operacional do pipeline (control table, logs)';


In [0]:
# auto-discovery + geração da control table

from pyspark.sql import functions as F

# 1. lista as tabelas da fonte
tables = (spark.read.format("jdbc")
    .option("url", url).option("user", user).option("password", pwd)
    .option("driver", "org.postgresql.Driver")
    .option("query", """
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_type = 'BASE TABLE'
          AND table_schema NOT IN ('pg_catalog','information_schema')
    """).load())

# 2. lê as colunas p/ detectar candidato a watermark (colunas de auditoria)
cols = (spark.read.format("jdbc")
    .option("url", url).option("user", user).option("password", pwd)
    .option("driver", "org.postgresql.Driver")
    .option("query", """
        SELECT table_schema, table_name, column_name
        FROM information_schema.columns
        WHERE table_schema NOT IN ('pg_catalog','information_schema')
    """).load())

# heurística: coluna cujo nome contém 'updat' ou 'modif' vira watermark
cand = (cols
    .filter(F.lower("column_name").rlike("updat|modif"))
    .groupBy("table_schema","table_name")
    .agg(F.first("column_name").alias("watermark_column")))

# 3. monta a control table com defaults sensatos
control = (tables
    .join(cand, ["table_schema","table_name"], "left")
    .withColumnRenamed("table_schema","source_schema")
    .withColumnRenamed("table_name","source_table")
    .withColumn("target_folder",  F.col("source_table"))
    .withColumn("load_type",      F.when(F.col("watermark_column").isNotNull(), F.lit("incremental"))
                                   .otherwise(F.lit("full")))
    .withColumn("ingestion_mode", F.lit("copy_into"))   # default que decidimos
    .withColumn("is_active",      F.lit(True))
    .select("source_schema","source_table","target_folder",
            "load_type","watermark_column","ingestion_mode","is_active"))

control.write.mode("overwrite").saveAsTable("hpn.metadata.control_ingestion")
display(control)


In [0]:
# Transformando a tabela de controle em JSON para facilitar a leitura do ADF
import json

rows = [r.asDict() for r in
        spark.table("hpn.metadata.control_ingestion")
             .filter("is_active = true")
             .collect()]

path = "abfss://landing@adlshpn.dfs.core.windows.net/_config/control_ingestion.json"
dbutils.fs.put(path, json.dumps(rows, indent=2, default=str), overwrite=True)

print(f"Exportei {len(rows)} tabelas -> {path}")
print(json.dumps(rows[:2], indent=2, default=str))   # espia as 2 primeiras
